# 03 · RQ3: inference batch size vs throughput, latency and memory (P4)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Syndr0/compute-efficient-genqa/blob/main/notebooks/03_rq3_inference_efficiency.ipynb)

Decodes the same 256 SQuAD prompts with the same fine-tuned model (merged LoRA; by default the
largest-budget multi-source adapter, otherwise the base model) under controlled configurations:

* batch size 1 → 64 (length-sorted batches, KV cache on)
* unsorted batches (padding cost)
* KV cache off (optional experiment, smaller subset)
* fp32 weights vs fp16/bf16

Each configuration: one warm-up batch, 3 timed repeats; quality (EM/F1) is recorded to confirm
it does not change. Run notebook 01 first so an adapter exists. Runtime on a T4: ~20 min.

In [ ]:
#@title Setup: clone the repo, install, choose where outputs go
#@markdown Outputs (data, adapters, predictions, metrics) persist on Google Drive so a
#@markdown disconnected runtime can resume: every stage skips work that is already saved.
USE_DRIVE = True  #@param {type:"boolean"}
SMOKE_TEST = False  #@param {type:"boolean"}
import os, sys
REPO_DIR = "/content/compute-efficient-genqa"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/Syndr0/compute-efficient-genqa.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -e .
sys.path.insert(0, f"{REPO_DIR}/src")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/cegqa_outputs"
else:
    OUT = "/content/cegqa_outputs"
if SMOKE_TEST:
    OUT += "_smoke"
os.environ["CEGQA_OUTPUT_DIR"] = OUT
CONFIG = "--config configs/default.yaml" + (" --config configs/smoke.yaml" if SMOKE_TEST else "")
print("outputs ->", OUT)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!python -m cegqa.cli prepare $CONFIG

In [ ]:
!python -m cegqa.cli rq3 $CONFIG

In [ ]:
from cegqa.utils import load_config
cfgs = ["configs/default.yaml"] + (["configs/smoke.yaml"] if SMOKE_TEST else [])
cfg = load_config(cfgs)

In [ ]:
import pandas as pd
from cegqa.utils import read_jsonl
df = pd.DataFrame(read_jsonl(f"{cfg['output_dir']}/rq3/results.jsonl"))
cols = ["group", "batch_size", "dtype_resolved", "examples_per_s", "gen_tokens_per_s", "batch_latency_mean_s",
        "batch_latency_p95_s", "peak_mem_gb", "padding_fraction", "em", "f1", "oom"]
df[[c for c in cols if c in df]].round(3)

In [ ]:
!python -m cegqa.cli report $CONFIG
from IPython.display import Image, display
for f in ["rq3_batch_sweep.png", "rq3_tradeoff.png"]:
    display(Image(f"{cfg['output_dir']}/figures/{f}"))